In [1]:
# 分析対象銘柄の証券コードをセット
from datetime import date
code = 2267
valuation_date = date.today()
valuation_date = date(2022, 11, 18)

In [2]:
# 読み込みファイルパスの設定とimportしたいmoduleパス(pythonパス)の設定
from pathlib import Path
import os

CURRENT_DIR = Path(os.getcwd())
PJ_DIR = CURRENT_DIR.parent.parent
DATA_DIR = PJ_DIR / "data" 


from wequant.data_processing import KessanPl, read_data
import polars as pl

In [3]:
# valuation_dateで指定した日の最新通期決算と決算予想をpl.DataFrameで返す
# def get_df_latest_yearly_performance(code: int, valuation_date: date=date.today()) -> pl.DataFrame:
fp = DATA_DIR/"kessan.parquet"
df = read_data(fp)
KPL = KessanPl(df)
KPL.with_columns_financtial_period()

df1 = KPL.get_latest_yearly_settlements(reference_date=valuation_date, settlement_type="予")
df1 = df1.with_columns([
  (pl.col("決算期") + pl.lit("(予)")).alias("決算期")
])

df2 = KPL.get_latest_yearly_settlements(reference_date=valuation_date, settlement_type="本")
df = pl.concat([df1, df2])
selected_cols = [df.columns[-1]] + df.columns[3:10]
df = df.filter(pl.col("code")==code)\
    .select(selected_cols)

rename_map_dct = {
    "announcement_date": "決算発表日",
    "sales": "売上高",
    "operating_income": "営業利益",
    "ordinary_profit": "経常利益",
    "final_profit": "純利益",
    "reviced_eps": "EPS",
    "dividend": "1株配当"
}
df = df.rename(rename_map_dct)

df

決算期,決算発表日,売上高,営業利益,経常利益,純利益,EPS,1株配当
str,date,i64,i64,i64,i64,f64,f64
"""2023年3月期(予)""",2022-11-11,481000,64000,79000,50000,320.5,90.0
"""2022年3月期""",2022-05-13,415116,53202,68549,44917,280.4,72.0


In [4]:
df.row(1)[0]

'2022年3月期'

In [5]:
a = '2024年3月期'
a

'2024年3月期'